# D4BL Training Pipeline — Qwen 3.5-4B

Run the headless training script on Colab A100. Trains 3 LoRA adapters (query parser, explainer, evaluator) with domain adaptation, then exports to GGUF. All outputs are backed up to Google Drive.

**Runtime:** Change to **A100** GPU via Runtime > Change runtime type.

**Time estimate:** ~30-50 minutes on A100.

## 0. Mount Google Drive

Mounts Drive for backup/restore. If a previous training run exists on Drive, it will be restored so the script can resume.

In [ ]:
from google.colab import drive
import shutil, os

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/d4bl_training_v3"
LOCAL_DIR = "/content/d4bl_training"

# Restore from Drive if a previous run exists
if os.path.exists(DRIVE_DIR):
    print(f"Found existing training state on Drive: {DRIVE_DIR}")
    shutil.copytree(DRIVE_DIR, LOCAL_DIR, dirs_exist_ok=True)
    items = sum(1 for _, _, files in os.walk(LOCAL_DIR) for _ in files)
    print(f"Restored {items} files — training script will resume from checkpoints")
else:
    print("No previous training state on Drive — starting fresh")

## 1. Verify GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Install dependencies

In [ ]:
import subprocess, sys

# Unsloth (includes transformers, peft, trl, etc.)
subprocess.run([sys.executable, "-m", "pip", "install", "unsloth"], check=True)

# HuggingFace datasets
subprocess.run([sys.executable, "-m", "pip", "install", "datasets"], check=True)

# Flash Linear Attention — accelerates Qwen 3.5's Gated Delta Networks
# Pure Python/Triton package, no CUDA build needed. causal-conv1d is optional.
subprocess.run([sys.executable, "-m", "pip", "install", "flash-linear-attention"], check=True)

## 3. Clone repo and set up

In [ ]:
import os
os.chdir("/content")

# Clone if not already present
if not os.path.exists("/content/d4bl_ai_agent"):
    !git clone https://github.com/William-Hill/d4bl_ai_agent.git

os.chdir("/content/d4bl_ai_agent")
!git pull origin main
!pwd

## 4. (Optional) HuggingFace token for gated models

In [ ]:
# Uncomment and set if needed for gated model access
# import os
# os.environ["HF_TOKEN"] = "hf_your_token_here"

## 5. Load training data and run

First cell copies training data from Drive. Second cell runs the full pipeline. If interrupted, rerun — it resumes from checkpoints.

In [ ]:
import shutil, os

# Copy training data from Drive if not already in repo
DATA_DIR = "/content/d4bl_ai_agent/scripts/training_data/final"
DRIVE_DATA = "/content/drive/MyDrive/d4bl_training_v3/training_data_final"

if not os.path.exists(DATA_DIR) or not os.listdir(DATA_DIR):
    if os.path.exists(DRIVE_DATA):
        os.makedirs(DATA_DIR, exist_ok=True)
        shutil.copytree(DRIVE_DATA, DATA_DIR, dirs_exist_ok=True)
        print(f"Copied training data from Drive ({len(os.listdir(DATA_DIR))} files)")
    else:
        print(f"ERROR: No training data found at {DRIVE_DATA}")
        print("Upload your scripts/training_data/final/ folder to MyDrive/d4bl_training_v3/training_data_final/")
else:
    print(f"Training data already present ({len(os.listdir(DATA_DIR))} files)")

!ls -lh {DATA_DIR}/

In [ ]:
import os

DATA_DIR = "/content/d4bl_ai_agent/scripts/training_data/final"

def count_lines(path):
    with open(path) as f:
        return sum(1 for _ in f)

files = [
    ("corpus_pretrain", "corpus_pretrain.jsonl"),
    ("query_parser (train)", "query_parser_train.jsonl"),
    ("query_parser (val)", "query_parser_val.jsonl"),
    ("explainer (train)", "explainer_train.jsonl"),
    ("explainer (val)", "explainer_val.jsonl"),
    ("evaluator (train)", "evaluator_train.jsonl"),
    ("evaluator (val)", "evaluator_val.jsonl"),
]

print("Training data counts:")
for label, filename in files:
    path = os.path.join(DATA_DIR, filename)
    if os.path.exists(path):
        print(f"  {label:<25}: {count_lines(path):>5} examples")
    else:
        print(f"  {label:<25}: MISSING")

In [ ]:
%env PYTHONPATH=/content/d4bl_ai_agent
!cd /content/d4bl_ai_agent && python scripts/training/train.py \
    --data-dir scripts/training_data/final \
    --output-dir /content/d4bl_training

In [ ]:
import shutil

DRIVE_DIR = "/content/drive/MyDrive/d4bl_training_v3"
LOCAL_DIR = "/content/d4bl_training"

print("Backing up training outputs to Google Drive...")
shutil.copytree(LOCAL_DIR, DRIVE_DIR, dirs_exist_ok=True)
items = sum(1 for _, _, files in os.walk(DRIVE_DIR) for _ in files)
print(f"Backed up {items} files to {DRIVE_DIR}")

## 7. Review results

In [ ]:
# Training report
with open("/content/d4bl_training/training_report.md") as f:
    from IPython.display import Markdown, display
    display(Markdown(f.read()))

In [ ]:
# List GGUF exports
!ls -lh /content/d4bl_training/gguf/*/

## 8. Download GGUFs

Download the GGUF files to use locally with Ollama. (Also available on Drive at `MyDrive/d4bl_training_v3/gguf/`)

In [ ]:
import shutil, os
from google.colab import files

GGUF_DIR = "/content/d4bl_training/gguf"
DOWNLOAD_DIR = "/content/d4bl_gguf_download"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# Map adapter dirs to expected filenames
ADAPTERS = {
    "d4bl-query-parser-qwen35": "d4bl-query-parser-qwen35-q4_k_m.gguf",
    "d4bl-explainer-qwen35": "d4bl-explainer-qwen35-q4_k_m.gguf",
    "d4bl-evaluator-qwen35": "d4bl-evaluator-qwen35-q4_k_m.gguf",
}

for adapter_prefix, target_name in ADAPTERS.items():
    # Unsloth exports to *_gguf/ subdirectory with generic name
    gguf_subdir = os.path.join(GGUF_DIR, f"{adapter_prefix}-q4_k_m_gguf")
    source = os.path.join(gguf_subdir, "domain_merged.Q4_K_M.gguf")
    dest = os.path.join(DOWNLOAD_DIR, target_name)
    if os.path.exists(source):
        shutil.copy2(source, dest)
        size_gb = os.path.getsize(dest) / (1024**3)
        print(f"{target_name} ({size_gb:.2f} GB)")
    else:
        print(f"WARNING: {source} not found")

print(f"\nDownloading from {DOWNLOAD_DIR}...")
for f in sorted(os.listdir(DOWNLOAD_DIR)):
    if f.endswith(".gguf"):
        files.download(os.path.join(DOWNLOAD_DIR, f))